In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_clinica")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyecto2404")

In [0]:
%python
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/citas.csv"

In [0]:
%python
citas_schema = StructType(fields=[
    StructField("id_cita", IntegerType(), True),
    StructField("fecha_programada", TimestampType(), True),
    StructField("id_paciente", IntegerType(), True),
    StructField("id_medico", IntegerType(), True),
    StructField("id_especialidad", IntegerType(), True),
    StructField("estado", StringType(), True),
    StructField("motivo_cancelacion", StringType(), True)
])

In [0]:
%python
df_citas = spark.read.option("header", True)\
                     .schema(citas_schema)\
                     .csv(ruta)

In [0]:
%python
df_citas_ingestion = df_citas.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
%python
df_citas_final = df_citas_ingestion.select(
    "id_cita",
    "fecha_programada",
    "id_paciente",
    "id_medico",
    "id_especialidad",
    "estado",
    "motivo_cancelacion",
    "INGESTION_DATE"
)

In [0]:
%python
df_citas_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.citas")